# Benchmark Phi-4-mini sur Kaggle (prompts maison + HumanEval)

**Avant de lancer** : panneau de droite > Session options > Accelerator = GPU, Internet = On.

Lancez les cellules **dans l'ordre**, une par une, et attendez la fin de chacune.

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # un seul GPU, meme si Kaggle en propose deux

REPO_URL = "https://github.com/FatiAllali123/SLM-Software-Engineering.git"
REPO_NAME = "SLM-Software-Engineering"
BRANCH = "correctifs-gemma"   

if not os.path.exists("../config.json"):
    !git clone -b {BRANCH} {REPO_URL}
    %cd {REPO_NAME}/benchmarking/notebooks

os.makedirs("../results", exist_ok=True)
print("Dossier courant :", os.getcwd())
!ls ..

Cloning into 'SLM-Software-Engineering'...
remote: Enumerating objects: 53, done.
remote: Counting objects: 100% (53/53), done.
remote: Compressing objects: 100% (34/34), done.
remote: Total 53 (delta 14), reused 26 (delta 9), pack-reused 0 (from 0)
Receiving objects: 100% (53/53), 35.16 KiB | 3.52 MiB/s, done.
Resolving deltas: 100% (14/14), done.
/kaggle/working/SLM-Software-Engineering/benchmarking/notebooks
Dossier courant : /kaggle/working/SLM-Software-Engineering/benchmarking/notebooks
config.json	       merge_results.py  README.md	   utils
grille_notation.md     notebooks	 requirements.txt
humaneval_subset.json  prompts.json	 results


In [2]:
!pip install -r ../requirements.txt -q
!nvidia-smi -L

GPU 0: Tesla T4 (UUID: GPU-3c4f1ce7-46dd-16f4-2211-2e3aff1b5324)
GPU 1: Tesla T4 (UUID: GPU-21b87fcc-d1c3-a80d-5edf-338c4839fd21)


In [3]:
import json, time, sys, re
import torch
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset

sys.path.append("..")
from utils.safe_exec import run_tests
from utils.extract import extract_code

# === MODELE : "phi" pour vous ===
MODEL_KEY = "phi"
# =================================

with open("../config.json") as f:
    config = json.load(f)

MODEL_NAME = config["models"][MODEL_KEY]
GEN_PARAMS = config["generation_params"]
HE_CONFIG = config["humaneval"]
SYSTEM_PROMPT = config["no_think_models"].get(MODEL_KEY)
model_slug = MODEL_NAME.split("/")[-1].lower()
GPU_NAME = torch.cuda.get_device_name(0)
print(MODEL_NAME, "|", model_slug, "|", GPU_NAME, "| precision config :", config["precision"])

microsoft/Phi-4-mini-instruct | phi-4-mini-instruct | Tesla T4 | precision config : bf16


In [4]:
torch.manual_seed(42)
DTYPE = torch.bfloat16 if config["precision"] == "bf16" else torch.float16
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, device_map="auto", torch_dtype=DTYPE)
model.eval()
print("Precision utilisee :", config["precision"])


def generate(user_prompt: str):
    messages = []
    if SYSTEM_PROMPT:
        messages.append({"role": "system", "content": SYSTEM_PROMPT})
    messages.append({"role": "user", "content": user_prompt})

    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True,
        return_tensors="pt", return_dict=True
    ).to(model.device)
    n_in = inputs["input_ids"].shape[1]

    # TTFT : temps pour produire 1 seul token
    torch.cuda.synchronize(); t0 = time.time()
    model.generate(**inputs, max_new_tokens=1, do_sample=False)
    torch.cuda.synchronize(); ttft = time.time() - t0

    # Generation complete
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize(); t0 = time.time()
    out = model.generate(**inputs, max_new_tokens=GEN_PARAMS["max_new_tokens"],
                         do_sample=GEN_PARAMS["do_sample"])
    torch.cuda.synchronize(); latency = time.time() - t0

    new_tokens = out[0][n_in:]
    return {
        "response": tokenizer.decode(new_tokens, skip_special_tokens=True),
        "gpu": GPU_NAME,
        "ttft_s": round(ttft, 3),
        "latency_s": round(latency, 3),
        "tokens_per_sec": round(len(new_tokens) / latency, 2) if latency > 0 else None,
        "input_tokens": n_in,
        "output_tokens": len(new_tokens),
        "gpu_memory_go": round(torch.cuda.max_memory_allocated() / 1e9, 2),
    }


# Echauffement + controle : le 1er appel est lent, et on verifie que le modele repond vraiment
test = generate("Write a Python function add(a, b) that returns a + b.")
print(test["response"][:300])
assert "def" in test["response"], "Reponse vide ou incoherente : prevenez le groupe avant de continuer."
print("Modele pret.")

config.json:   0%|          | 0.00/2.50k [00:00<?, ?B/s]

[transformers] This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.


tokenizer_config.json:   0%|          | 0.00/2.93k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 15.5MB            

tokenizer.json: downloading bytes:           |  0.00B            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/194 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

Precision utilisee : bf16
def add(a, b):
    return a + b

# Example usage:
result = add(3, 5)
print(result)  # Output: 8

Modele pret.


## Partie A : 15 prompts maison

In [5]:
with open("../prompts.json") as f:
    prompts = json.load(f)["prompts"]

def norm(s):
    return re.sub(r"\s+", "", s)

results = []
for p in prompts:
    gen = generate(p["prompt"])
    response = gen.pop("response")

    status, error = None, None
    if p["task"] == "generation_code" and "test" in p:
        status, error = run_tests(extract_code(response), p["test"],
                                  timeout=HE_CONFIG["timeout_seconds"])
    elif p["task"] == "completion_code" and "expected" in p:
        status = "pass" if norm(p["expected"]) in norm(response) else "fail"

    results.append({
        "id": p["id"], "task": p["task"], "prompt": p["prompt"], "response": response,
        **gen,
        "auto_status": status, "auto_error": error,
        "bug_label": None, "note_qualite": None, "hallucination": None,
    })
    print(p["id"], status)

df_prompts = pd.DataFrame(results)
df_prompts.to_csv(f"../results/benchmark_{model_slug}.csv", index=False)
print(f"-> results/benchmark_{model_slug}.csv")
df_prompts[["id", "task", "ttft_s", "latency_s", "tokens_per_sec", "gpu_memory_go", "auto_status"]]

P1 fail
Le 10-ième terme de la suite de Fibonacci est: 55
P2 pass
monde le Bonjour
P3 pass
Le nombre de voyelles dans la chaîne est: 3
P4 pass
P5 pass
P6 pass
P7 None
P8 None
P9 None
P10 None
P11 None
P12 None
P13 None
P14 None
P15 None
-> results/benchmark_phi-4-mini-instruct.csv


,id,task,ttft_s,latency_s,tokens_per_sec,gpu_memory_go,auto_status
0,P1,generation_code,0.295,2.712,18.81,7.69,fail
1,P2,generation_code,0.187,6.432,19.75,7.71,pass
2,P3,generation_code,0.187,3.802,19.20,7.70,pass
3,P4,generation_code,0.189,4.504,19.76,7.70,pass
4,P5,completion_code,0.191,1.545,18.77,7.69,pass
5,P6,completion_code,0.196,6.954,19.99,7.71,pass
6,P7,bug_detection,0.191,6.043,19.86,7.71,None
7,P8,bug_detection,0.198,7.482,19.92,7.72,None
8,P9,bug_detection,0.195,6.093,19.69,7.71,None
9,P10,explication,0.208,1.462,17.79,7.70,None


## Partie B : HumanEval (20 problemes communs)

In [6]:
with open("../humaneval_subset.json") as f:
    subset = json.load(f)

humaneval = load_dataset(HE_CONFIG["dataset"], split="test")
humaneval = humaneval.filter(lambda ex: ex["task_id"] in subset["task_ids"])
print(len(humaneval), "problemes")

HEADER = "from typing import *\nimport math\n\n"

details = []
for ex in humaneval:
    gen = generate(HE_CONFIG["instruction"] + ex["prompt"])
    code_generated = extract_code(gen["response"])
    test_code = ex["test"] + f"\ncheck({ex['entry_point']})"
    # methode alignee sur le notebook Gemma : enonce + code du modele
    status, error = run_tests(HEADER + ex["prompt"] + "\n" + code_generated, test_code,
                              timeout=HE_CONFIG["timeout_seconds"])
    details.append({
        "task_id": ex["task_id"], "status": status, "error": error,
        "ttft_s": gen["ttft_s"], "latency_s": gen["latency_s"],
        "tokens_per_sec": gen["tokens_per_sec"],
        "output_tokens": gen["output_tokens"], "gpu_memory_go": gen["gpu_memory_go"],
        "response": gen["response"],
    })
    print(ex["task_id"], status)

n_pass = sum(d["status"] == "pass" for d in details)
summary = {
    "model": MODEL_NAME, "gpu": GPU_NAME, "precision": config["precision"],
    "dataset": "HumanEval (sous-ensemble des 20 premiers problemes)",
    "n_problems": len(details), "n_pass": n_pass,
    "pass@1": round(100 * n_pass / len(details), 1),
    "details": details,
}
with open(f"../results/humaneval_{model_slug}.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print(f"pass@1 = {summary['pass@1']}% ({n_pass}/{len(details)}) -> results/humaneval_{model_slug}.json")

README.md:   0%|          | 0.00/6.52k [00:00<?, ?B/s]

openai_humaneval/test-00000-of-00001.par(…): reconstructing file:   0%|          |  0.00B / 83.9kB            

openai_humaneval/test-00000-of-00001.par(…): downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/164 [00:00<?, ? examples/s]

Filter:   0%|          | 0/164 [00:00<?, ? examples/s]

20 problemes
HumanEval/0 pass
HumanEval/1 fail
HumanEval/2 pass
HumanEval/3 pass
HumanEval/4 pass
HumanEval/5 pass
HumanEval/6 pass
HumanEval/7 pass
HumanEval/8 pass
HumanEval/9 pass
HumanEval/10 fail
HumanEval/11 pass
HumanEval/12 pass
HumanEval/13 pass
HumanEval/14 pass
HumanEval/15 pass
HumanEval/16 pass
HumanEval/17 fail
HumanEval/18 pass
HumanEval/19 pass
pass@1 = 85.0% (17/20) -> results/humaneval_phi-4-mini-instruct.json


In [7]:
from importlib.metadata import version
import platform
lines = [f"# Environnement - {MODEL_NAME}", "",
         "- Plateforme : Kaggle",
         f"- GPU : {GPU_NAME}",
         f"- Precision : {config['precision']}",
         f"- Python : {platform.python_version()}",
         f"- torch : {torch.__version__}",
         f"- CUDA : {torch.version.cuda}"]
for lib in ["transformers", "accelerate", "huggingface_hub", "datasets", "pandas"]:
    lines.append(f"- {lib} : {version(lib)}")
open(f"../environnement_{model_slug}.md", "w").write("\n".join(lines) + "\n")
print("\n".join(lines))
print()
!ls -la ../results ../environnement_*.md

# Environnement - microsoft/Phi-4-mini-instruct

- Plateforme : Kaggle
- GPU : Tesla T4
- Precision : bf16
- Python : 3.13.15
- torch : 2.11.0+cu128
- CUDA : 12.8
- transformers : 5.16.1
- accelerate : 1.14.0
- huggingface_hub : 1.29.0
- datasets : 4.8.5
- pandas : 2.3.3

-rw-r--r-- 1 root root  272 Oct 25 21:56 ../environnement_phi-4-mini-instruct.md

../results:
total 44
drwxr-xr-x 2 root root  4096 Oct 25 21:53 .
drwxr-xr-x 5 root root  4096 Oct 25 21:56 ..
-rw-r--r-- 1 root root 11549 Oct 25 21:50 benchmark_phi-4-mini-instruct.csv
-rw-r--r-- 1 root root     1 Oct 25 21:45 .gitkeep
-rw-r--r-- 1 root root 17038 Oct 25 21:53 humaneval_phi-4-mini-instruct.json
